In [1]:
import torch as t
from torch import Tensor
from jaxtyping import Float

from nnsight import LanguageModel

import sys

sys.path.append("../../../")

from nngine import alter

device = "cuda"

In [2]:
model = LanguageModel(
    'openai-community/gpt2',
    device_map="cuda:0",
    dispatch=True
)
tokenizer = model.tokenizer

alter(model)

In [3]:
from ioi_dataset import IOIDataset

N = 25
clean_dataset = IOIDataset(
    prompt_type='mixed',
    N=N,
    tokenizer=model.tokenizer,
    prepend_bos=False,
    seed=1,
    device=device
)
corr_dataset = clean_dataset.gen_flipped_prompts('ABC->XYZ, BAB->XYZ')

In [4]:
def ave_logit_diff(
    logits: Float[Tensor, 'batch seq d_vocab'],
    ioi_dataset: IOIDataset,
    per_prompt: bool = False
):
    '''
        Return average logit difference between correct and incorrect answers
    '''
    # Get logits for indirect objects
    batch_size = logits.size(0)
    io_logits = logits[range(batch_size), ioi_dataset.word_idx['end'][:batch_size], ioi_dataset.io_tokenIDs[:batch_size]]
    s_logits = logits[range(batch_size), ioi_dataset.word_idx['end'][:batch_size], ioi_dataset.s_tokenIDs[:batch_size]]
    # Get logits for subject
    logit_diff = io_logits - s_logits
    return logit_diff if per_prompt else logit_diff.mean()


with t.no_grad():
    with model.trace(clean_dataset.toks):
        clean_logits = model.output.logits.save()

    with model.trace(corr_dataset.toks):
        corrupt_logits = model.output.logits.save()

clean_logits = clean_logits.value
corrupt_logits = corrupt_logits.value

clean_logit_diff = ave_logit_diff(clean_logits, clean_dataset).item()
corrupt_logit_diff = ave_logit_diff(corrupt_logits, corr_dataset).item()

def ioi_metric(
    logits: Float[Tensor, "batch seq_len d_vocab"],
    corrupted_logit_diff: float = corrupt_logit_diff,
    clean_logit_diff: float = clean_logit_diff,
    ioi_dataset: IOIDataset = clean_dataset
 ):
    patched_logit_diff = ave_logit_diff(logits, ioi_dataset)
    return (patched_logit_diff - corrupted_logit_diff) / (clean_logit_diff - corrupted_logit_diff)

def negative_ioi_metric(logits: Float[Tensor, "batch seq_len d_vocab"]):
    return -ioi_metric(logits)
    
# Get clean and corrupt logit differences
with t.no_grad():
    clean_metric = ioi_metric(clean_logits, corrupt_logit_diff, clean_logit_diff, clean_dataset)
    corrupt_metric = ioi_metric(corrupt_logits, corrupt_logit_diff, clean_logit_diff, corr_dataset)

print(f'Clean direction: {clean_logit_diff}, Corrupt direction: {corrupt_logit_diff}')
print(f'Clean metric: {clean_metric}, Corrupt metric: {corrupt_metric}')

You're using a GPT2TokenizerFast tokenizer. Please note that with a fast tokenizer, using the `__call__` method is faster than using a method to encode the text followed by a call to the `pad` method to get a padded encoding.


Clean direction: 3.1688125133514404, Corrupt direction: 1.960614562034607
Clean metric: 1.0, Corrupt metric: 0.0


In [5]:
import new_eap

import importlib

importlib.reload(new_eap)

graph = new_eap.EAP({}, components=["head", "mlp"])

In [6]:
graph.run(
    model,
    clean_dataset.toks,
    corr_dataset.toks,
    batch_size=25,
    metric=ioi_metric,
)

You're using a GPT2TokenizerFast tokenizer. Please note that with a fast tokenizer, using the `__call__` method is faster than using a method to encode the text followed by a call to the `pad` method to get a padded encoding.


In [7]:
edges = graph.top_edges(n=20, format=True)

mlp.2 -> [1.606] -> mlp.3
mlp.0 -> [-0.951] -> mlp.1
mlp.2 -> [0.657] -> mlp.4
mlp.0 -> [0.347] -> mlp.2
mlp.2 -> [-0.343] -> mlp.8
mlp.10 -> [-0.29] -> mlp.11
mlp.5 -> [0.229] -> mlp.6
mlp.2 -> [-0.191] -> mlp.6
mlp.0 -> [-0.179] -> mlp.10
mlp.3 -> [0.164] -> mlp.5
mlp.2 -> [-0.155] -> mlp.9
mlp.1 -> [-0.149] -> mlp.2
mlp.4 -> [0.138] -> mlp.6
mlp.4 -> [0.129] -> mlp.5
head.0.4 -> [-0.128] -> mlp.0
mlp.0 -> [-0.125] -> mlp.4
mlp.2 -> [-0.123] -> mlp.10
mlp.3 -> [0.118] -> mlp.10
head.1.9 -> [-0.116] -> mlp.1
mlp.0 -> [0.114] -> mlp.5
